In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


csv_files = {}

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            csv_files[file] = os.path.join(root, file)

print("Files found:")
for name, path in csv_files.items():
    print(name, "->", path)


train = pd.read_csv(csv_files["train.csv"])
test = pd.read_csv(csv_files["test.csv"])

print("\nTrain shape:", train.shape)
print("Test shape:", test.shape)


target_candidates = list(set(train.columns) - set(test.columns))

print("\nPossible target column:", target_candidates)

TARGET = target_candidates[0]

print("TARGET =", TARGET)


X = train.drop(columns=[TARGET])
y = train[TARGET]


numeric_features = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("\nNumerical columns:", numeric_features)
print("Categorical columns:", categorical_features)

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])


X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

Files found:
sample_submission.csv -> /kaggle/input/competitions/code-forge-club-ml-challenge-diabetes-risk-prediction-it/sample_submission.csv
data_dictionary.csv -> /kaggle/input/competitions/code-forge-club-ml-challenge-diabetes-risk-prediction-it/data_dictionary.csv
train.csv -> /kaggle/input/competitions/code-forge-club-ml-challenge-diabetes-risk-prediction-it/train.csv
test.csv -> /kaggle/input/competitions/code-forge-club-ml-challenge-diabetes-risk-prediction-it/test.csv

Train shape: (10000, 16)
Test shape: (2000, 15)

Possible target column: ['Complication_Risk']
TARGET = Complication_Risk

Numerical columns: ['Age', 'BMI', 'Fasting_Glucose_mg_dL', 'HbA1c_percent', 'Systolic_BP_mmHg', 'Diastolic_BP_mmHg', 'Total_Cholesterol_mg_dL', 'Diabetes_Duration_years', 'Comorbidity_Count']
Categorical columns: ['Patient_ID', 'Gender', 'Physical_Activity', 'Smoking_Status', 'Family_History', 'Insulin_Use']
Training rows: 8000
Validation rows: 2000


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)


model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)


pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])


pipeline.fit(X_train, y_train)


y_pred = pipeline.predict(X_valid)


accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(
    y_valid, y_pred, average="weighted", zero_division=0
)
recall = recall_score(
    y_valid, y_pred, average="weighted", zero_division=0
)
f1 = f1_score(
    y_valid, y_pred, average="weighted", zero_division=0
)

print("========== MODEL RESULTS ==========")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(y_valid, y_pred, zero_division=0))


pipeline.fit(X, y)


test_predictions = pipeline.predict(test)

print("\nTest predictions created:", len(test_predictions))


sample_path = csv_files["sample_submission.csv"]
sample_submission = pd.read_csv(sample_path)

print("\nSample submission columns:")
print(sample_submission.columns.tolist())


prediction_columns = list(
    set(sample_submission.columns) - set(test.columns)
)

print("Prediction column:", prediction_columns)

PREDICTION_COLUMN = prediction_columns[0]


submission = sample_submission.copy()
submission[PREDICTION_COLUMN] = test_predictions


submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print("\n submission.csv created successfully!")
display(submission.head())

========== MODEL RESULTS ==========
Accuracy : 0.753
Precision: 0.7199970406003038
Recall   : 0.753
F1 Score : 0.7352706511175898

Classification Report:
              precision    recall  f1-score   support

        High       0.00      0.00      0.00        94
         Low       0.79      0.77      0.78       935
    Moderate       0.72      0.81      0.76       971

    accuracy                           0.75      2000
   macro avg       0.50      0.53      0.51      2000
weighted avg       0.72      0.75      0.74      2000


Test predictions created: 2000

Sample submission columns:
['Patient_ID', 'Complication_Risk']
Prediction column: ['Complication_Risk']

 submission.csv created successfully!


,Patient_ID,Complication_Risk
0,P10001,Low
1,P10002,Low
2,P10003,Moderate
3,P10004,Low
4,P10005,Low
